# TalentLens
Extracción de información de CVs con PLN para agilizar la preselección de candidatos.

## 1. Captura de información
Leemos el texto de los CVs en PDF y Word. Los PDF escaneados se leen con OCR.

In [7]:
from pathlib import Path

import pymupdf
import docx
import pytesseract
import pandas as pd
from PIL import Image

DATA = Path('CVs_ficticios_prueba')
archivos = sorted(list(DATA.glob('pdf/*.pdf')) + list(DATA.glob('word/*.docx')))

print(len(archivos), 'CVs encontrados')
print('Tesseract', pytesseract.get_tesseract_version())
print('Idiomas OCR:', pytesseract.get_languages())

20 CVs encontrados
Tesseract 5.5.3
Idiomas OCR: ['afr', 'amh', 'ara', 'asm', 'aze', 'aze_cyrl', 'bel', 'ben', 'bod', 'bos', 'bre', 'bul', 'cat', 'ceb', 'ces', 'chi_sim', 'chi_sim_vert', 'chi_tra', 'chi_tra_vert', 'chr', 'cos', 'cym', 'dan', 'deu', 'div', 'dzo', 'ell', 'eng', 'enm', 'epo', 'equ', 'est', 'eus', 'fao', 'fas', 'fil', 'fin', 'fra', 'frk', 'frm', 'fry', 'gla', 'gle', 'glg', 'grc', 'guj', 'hat', 'heb', 'hin', 'hrv', 'hun', 'hye', 'iku', 'ind', 'isl', 'ita', 'ita_old', 'jav', 'jpn', 'jpn_vert', 'kan', 'kat', 'kat_old', 'kaz', 'khm', 'kir', 'kmr', 'kor', 'kor_vert', 'lao', 'lat', 'lav', 'lit', 'ltz', 'mal', 'mar', 'mkd', 'mlt', 'mon', 'mri', 'msa', 'mya', 'nep', 'nld', 'nor', 'oci', 'ori', 'osd', 'pan', 'pol', 'por', 'pus', 'que', 'ron', 'rus', 'san', 'sin', 'slk', 'slv', 'snd', 'spa', 'spa_old', 'sqi', 'srp', 'srp_latn', 'sun', 'swa', 'swe', 'syr', 'tam', 'tat', 'tel', 'tgk', 'tha', 'tir', 'ton', 'tur', 'uig', 'ukr', 'urd', 'uzb', 'uzb_cyrl', 'vie', 'yid', 'yor']


### Funciones de lectura
- **PDF:** texto con pymupdf; si la página no tiene texto (escaneada) se usa OCR en español (`--psm 4` mantiene juntas las filas de las tablas).
- **Word:** párrafos y tablas en el orden del documento.

In [12]:
def leer_pdf(ruta):
    """Texto de un PDF; si una página no tiene texto, se lee con OCR."""
    paginas = []
    with pymupdf.open(ruta) as pdf:
        for pagina in pdf:
            texto = pagina.get_text()
            if len(texto.strip()) < 20:
                pix = pagina.get_pixmap(dpi=300)
                img = Image.frombytes('RGB', (pix.width, pix.height), pix.samples)
                texto = pytesseract.image_to_string(img, lang='spa', config='--psm 4')
            paginas.append(texto)
    return '\n'.join(paginas)

def leer_docx(ruta):
    """Texto de un Word en orden, incluyendo tablas."""
    doc = docx.Document(ruta)
    lineas = []
    for bloque in doc.iter_inner_content():
        if isinstance(bloque, docx.table.Table):
            for fila in bloque.rows:
                celdas = []
                for celda in fila.cells:
                    if celda.text not in celdas:  # celdas combinadas se repiten
                        celdas.append(celda.text)
                lineas.append('\n'.join(celdas))
        else:
            lineas.append(bloque.text)
    return '\n'.join(lineas)

def leer_cv(ruta):
    ruta = Path(ruta)
    if ruta.suffix.lower() == '.pdf':
        return leer_pdf(ruta)
    if ruta.suffix.lower() == '.docx':
        return leer_docx(ruta)
    raise ValueError(f'Formato no soportado: {ruta.suffix}')

### Leer todos los CVs

In [9]:
cvs = pd.DataFrame({
    'archivo': [str(a.relative_to(DATA)) for a in archivos],
    'texto': [leer_cv(a) for a in archivos],
})
cvs['caracteres'] = cvs['texto'].str.len()
cvs[['archivo', 'caracteres']]

,archivo,caracteres
0,pdf/cv_01_clasico.pdf,698
1,pdf/cv_02_moderno_foto.pdf,1052
2,pdf/cv_03_tabla.pdf,1060
3,pdf/cv_05_creativo_foto.pdf,840
4,pdf/cv_06_clasico.pdf,1205
5,pdf/cv_07_escaneado.pdf,888
6,pdf/cv_09_moderno_foto.pdf,753
7,pdf/cv_10_varias_paginas.pdf,2206
8,pdf/cv_11_creativo_foto.pdf,685
9,pdf/cv_13_tabla.pdf,1101


## 2. Limpieza
Decisiones:
- Normalizar unicode (NFC) y espacios.
- Quitar viñetas y unificar guiones largos.
- Unir correos partidos en dos líneas.
- Reparar la `@` mal leída por el OCR (solo dominios conocidos).
- Mantener mayúsculas: el NER las usa para detectar nombres.

In [13]:
import re
import unicodedata

VINETAS = '•·«»*▪●■◦–—-'
DOMINIOS = r'(gmail|yahoo|outlook|hotmail)\.com'

def limpiar_texto(texto):
    texto = unicodedata.normalize('NFC', texto)
    texto = texto.replace('\xa0', ' ').replace('\t', ' ')
    # correos partidos en dos líneas (columnas angostas): "outlook.\ncom"
    texto = re.sub(r'(\S+@\S+)\n([a-z.]{1,4})$', r'\1\2', texto, flags=re.M)
    # OCR: la @ se lee como "0", "QW" o "GW" -> pedro.xicara890gmail.com
    texto = re.sub(r'\b([a-z][\w.]*?)[0OQGW]{1,2}' + DOMINIOS, r'\1@\2.com', texto)
    lineas = []
    for linea in texto.split('\n'):
        linea = linea.strip().lstrip(VINETAS).strip()  # viñetas al inicio
        linea = re.sub(r'\s*[–—]\s*', ' - ', linea)      # guiones largos
        linea = re.sub(r' {2,}', ' ', linea)
        if linea:
            lineas.append(linea)
    return '\n'.join(lineas)

In [14]:
cvs['limpio'] = cvs['texto'].map(limpiar_texto)
print(cvs.loc[1, 'limpio'][:400])

Contacto
+502 4260-9980
carlos.rodriguez86@outlook.com
Escuintla
Competencias
AWS
PostgreSQL
REST APIs
Python
Linux
Idiomas
Español (nativo)
K'iche' (intermedio)
Francés (básico)
Carlos Enrique Rodríguez López
Desarrollador Backend
Sobre mí
Desarrollador Backend con 9 años de experiencia. Me interesa aportar en equipos que
buscan resultados medibles y mejora continua.
Experiencia
Desarrollador Sen


### Separar secciones
Cada CV nombra sus secciones distinto (Experiencia / Trayectoria / Historial laboral, Skills / Aptitudes...). Las agrupamos con un diccionario de sinónimos.

Lo que está antes del primer encabezado queda como `encabezado`.

In [15]:
SECCIONES = {
    'contacto': ['contacto', 'datos personales', 'informacion de contacto'],
    'perfil': ['perfil', 'perfil profesional', 'sobre mi', 'resumen', 'objetivo', 'acerca de mi', 'profile'],
    'experiencia': ['experiencia', 'experiencia laboral', 'experiencia profesional', 'trayectoria',
                    'trayectoria profesional', 'historial laboral', 'experience'],
    'educacion': ['educacion', 'formacion', 'formacion academica', 'preparacion academica', 'estudios', 'education'],
    'habilidades': ['habilidades', 'competencias', 'skills', 'aptitudes', 'conocimientos',
                    'conocimientos tecnicos', 'herramientas y conocimientos'],
    'idiomas': ['idiomas', 'lenguas', 'languages'],
    'cursos': ['cursos', 'certificaciones', 'cursos y certificaciones'],
    'referencias': ['referencias'],
    'otros': ['informacion adicional'],
}
ENCABEZADOS = {frase: sec for sec, frases in SECCIONES.items() for frase in frases}
# segunda palabra de un encabezado partido en dos líneas (tablas): "EXPERIENCIA\nPROFESIONAL"
CONTINUACION = {'profesional', 'laboral', 'academica', 'tecnicos'}

def sin_acentos(s):
    s = unicodedata.normalize('NFD', s.lower())
    return ''.join(ch for ch in s if unicodedata.category(ch) != 'Mn')

def detectar_encabezado(linea):
    """Devuelve (sección, resto de la línea) o (None, línea)."""
    clave = sin_acentos(linea).strip(' :')
    if clave in ENCABEZADOS:
        return ENCABEZADOS[clave], ''
    # OCR de tablas: encabezado en mayúsculas pegado al contenido, "PERFIL Coordinador ..."
    palabras = linea.split()
    for n in (2, 1):
        inicio = ' '.join(palabras[:n])
        if inicio.isupper() and sin_acentos(inicio) in ENCABEZADOS and len(palabras) > n:
            return ENCABEZADOS[sin_acentos(inicio)], ' '.join(palabras[n:])
    return None, linea

def separar_secciones(texto):
    secciones = {'encabezado': []}
    actual = 'encabezado'
    anterior_fue_encabezado = False
    for linea in texto.split('\n'):
        primera = sin_acentos(linea.split()[0])
        if anterior_fue_encabezado and primera in CONTINUACION and linea.split()[0].isupper():
            linea = ' '.join(linea.split()[1:])
            if not linea:
                continue
        seccion, resto = detectar_encabezado(linea)
        anterior_fue_encabezado = seccion is not None
        if seccion:
            actual = seccion
            secciones.setdefault(actual, [])
            linea = resto
        if linea:
            secciones[actual].append(linea)
    return {sec: '\n'.join(lineas) for sec, lineas in secciones.items()}

In [16]:
cvs['secciones'] = cvs['limpio'].map(separar_secciones)

pd.DataFrame([{sec: '✓' for sec in s} for s in cvs['secciones']], index=cvs['archivo']).fillna('')

,encabezado,perfil,experiencia,educacion,habilidades,idiomas,contacto,cursos,referencias,otros
archivo,,,,,,,,,,
pdf/cv_01_clasico.pdf,✓,✓,✓,✓,✓,✓,,,,
pdf/cv_02_moderno_foto.pdf,✓,✓,✓,✓,✓,✓,✓,,,
pdf/cv_03_tabla.pdf,✓,✓,✓,✓,✓,✓,✓,,,
pdf/cv_05_creativo_foto.pdf,✓,✓,✓,✓,✓,✓,,,,
pdf/cv_06_clasico.pdf,✓,✓,✓,✓,✓,✓,,,,
pdf/cv_07_escaneado.pdf,✓,✓,✓,✓,✓,✓,,,,
pdf/cv_09_moderno_foto.pdf,✓,✓,✓,✓,✓,✓,✓,,,
pdf/cv_10_varias_paginas.pdf,✓,✓,✓,✓,✓,✓,,✓,✓,✓
pdf/cv_11_creativo_foto.pdf,✓,✓,✓,✓,✓,✓,,,,
